In [1]:
import pickle, os
import numpy as np
from collections import Counter
from datasets import Dataset
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset
from lowresource_llm_evaluation.generateDataset import quitarYaAnotadas, generateInstructivoQA, generateInstructivoDataset
from transformers import AutoTokenizer
import pandas as pd
from dotenv import load_dotenv
load_dotenv("secrets.env")

MODELO = "Qwen/Qwen2.5-7B-Instruct" 
TOKENIZER = AutoTokenizer.from_pretrained(MODELO, trust_remote_code=True)

def load_custom_txt(file_path, ds_instance, name="mi_txt_chat"):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Separamos por bloques, limpiamos espacios y filtramos vacíos
    # Usamos el delimitador de usuario para identificar cada "ejemplo"
    blocks = ["<|user|>" + b.strip() for b in content.split("<|user|>") if b.strip()]
    
    # Llamamos a tu método existente
    ds_instance.read_list(blocks, dataset_name=name)
    return ds_instance

# ---------------------------------------------------------
# 3. Función principal integrada
# ---------------------------------------------------------
def saveTrainTest(
    language,
    modelo_name, # Nombre para la carpeta
    tokenizer,
    thr=0.3,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    concatenate=False,
    max_tokens=1024,
    N_max_lengths=20000,
    max_length=None,
    min_words = 4,
    source= "NLLB"
):
    root_dir = "TrainDatasets"
    os.makedirs(root_dir, exist_ok=True)

    # El nombre del modelo suele tener '/', lo limpiamos para la carpeta
    modelo_folder = modelo_name.split("/")[-1]
    save_dir = f"{root_dir}/{modelo_folder}"
    os.makedirs(save_dir, exist_ok=True)

    # 1. Carga y filtrado inicial
    ds = (
        LanguageDataset(language, min_words=min_words)
        .read_opus(source=source, version=1)
        .filter_by_language(top_k=10, filter_language_thr=thr, batch_size=1024)
    )
    print("Estadísticas antes de concatenar")
    if compute_length:
        lengths = ds.get_stats(tokenizer, N_max=N_max_lengths)

    # 2. Concatenación (Nuevo método interno)
    if concatenate:
        ds.concatenate(tokenizer, max_tokens=max_tokens)
        if compute_length:
            print("Estadísticas después de concatenar")
            lengths = ds.get_stats(tokenizer, N_max=N_max_lengths)
    # 3. Estadísticas y Auto-ajuste de max_length
    # Si max_length es None, forzamos compute_length para saber el percentil 95
    if max_length is None and lengths is not None:
        max_length = int(np.percentile(lengths, 95))
        print(f"🎯 max_length ajustado automáticamente al P95: {max_length}")

    # 4. Split y Tokenización
    train = ds.tokenize(
        tokenizer=tokenizer,
        max_length=max_length
    )

    # 5. Guardar
    save_path = f"{save_dir}/{language}.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(train, f, protocol=5)
    

    print(f"\nDataset guardado: {save_path} Train: {len(train)}")
    return train

def saveInstructivo(
    language,
    modelo_name, # Nombre para la carpeta
    tokenizer,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=None
):
    root_dir = "TrainDatasets"
    os.makedirs(root_dir, exist_ok=True)

    # El nombre del modelo suele tener '/', lo limpiamos para la carpeta
    modelo_folder = modelo_name.split("/")[-1]
    save_dir = f"{root_dir}/{modelo_folder}"
    os.makedirs(save_dir, exist_ok=True)

    dsInstructivo = LanguageDataset(language).read_local_file("TrainDatasets",f"Instructivo/{language}.csv").read_local_file("TrainDatasets",f"Instructivo_QA/{language}.csv")
    if os.path.exists(f"/notebooks/TrainDatasets/Generados-Instructivos/{language}.txt"):
        load_custom_txt(f"/notebooks/TrainDatasets/Generados-Instructivos/{language}.txt", dsInstructivo, "Instructivos-Generados-GPT")

    if compute_length or max_length is None:
        lengths = dsInstructivo.get_stats(tokenizer, N_max=N_max_lengths)
        
        if max_length is None and lengths is not None:
            max_length = int(np.percentile(lengths, 95))
            print(f"🎯 max_length ajustado automáticamente al P95: {max_length}")
        
        
    instructivoResult = dsInstructivo.tokenize(
            tokenizer=tokenizer,
            max_length=max_length
        )

    save_path = f"{save_dir}/{language}-Instructivo.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(instructivoResult, f, protocol=5)
    print(f"\nDataset guardado: {save_path} Instructivo: {len(instructivoResult)}")
    return instructivoResult, dsInstructivo

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
language = "asturiano"
saveTrainTest(
    language,
    MODELO,
    TOKENIZER,
    thr=0.5,
    compute_length=True,
    concatenate=False,
    max_tokens=160,
    N_max_lengths=200000, 
    max_length=120
)
saveInstructivo(
    language,
    MODELO, # Nombre para la carpeta
    TOKENIZER,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=160
)

[INFO] Descargando FastText LID-176 a /usr/local/lib/python3.11/dist-packages/lowresource_llm_evaluation/models/lid.176.ftz ...
[INFO] Modelo FastText descargado correctamente.


Empezando descarga de NLLB...
Procesando líneas directamente desde el flujo comprimido...
Dataset cargado: 2721662 líneas nuevas.
Estadísticas antes de concatenar

📊 Analizando estadísticas de tokens (N=200000)...
------------------------------------------------
Total líneas en dataset: 528126
Media: 48.14 | Mediana: 42.00
Percentil 95: 100.00 (recomendado para max_length)
Percentil 98: 120.00
Máximo: 184 | Mínimo: 6
Moda (aprox): 30 tokens
------------------------------------------------



Map:   0%|          | 0/528126 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/asturiano.pkl Train: 528126
Leyendo archivo Instructivo/asturiano.csv...


Leyendo archivo Instructivo_QA/asturiano.csv...

📊 Analizando estadísticas de tokens (N=1223)...
------------------------------------------------
Total líneas en dataset: 1223
Media: 210.13 | Mediana: 112.00
Percentil 95: 740.90 (recomendado para max_length)
Percentil 98: 910.80
Máximo: 1453 | Mínimo: 20
Moda (aprox): 40 tokens
------------------------------------------------



Map:   0%|          | 0/1223 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/asturiano-Instructivo.pkl Instructivo: 1223


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1223
 }),
 <lowresource_llm_evaluation.LanguageDatasets.LanguageDataset at 0x7fec16341e90>)

In [3]:
language = "aranes"
saveTrainTest(
    language,
    MODELO,
    TOKENIZER,
    thr=0.5,
    compute_length=True,
    concatenate=False,
    max_tokens=160,
    N_max_lengths=200000, 
    max_length=160,
    min_words=40
)
saveInstructivo(
    language,
    MODELO, # Nombre para la carpeta
    TOKENIZER,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=160
)

Empezando descarga de NLLB...
Procesando líneas directamente desde el flujo comprimido...
Dataset cargado: 762894 líneas nuevas.
Estadísticas antes de concatenar

📊 Analizando estadísticas de tokens (N=22704)...
------------------------------------------------
Total líneas en dataset: 22704
Media: 104.77 | Mediana: 100.00
Percentil 95: 148.00 (recomendado para max_length)
Percentil 98: 159.00
Máximo: 195 | Mínimo: 57
Moda (aprox): 90 tokens
------------------------------------------------



Map:   0%|          | 0/22704 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/aranes.pkl Train: 22704
Leyendo archivo Instructivo/aranes.csv...


Leyendo archivo Instructivo_QA/aranes.csv...

📊 Analizando estadísticas de tokens (N=1006)...
------------------------------------------------
Total líneas en dataset: 1006
Media: 287.15 | Mediana: 186.00
Percentil 95: 934.00 (recomendado para max_length)
Percentil 98: 1101.00
Máximo: 2013 | Mínimo: 26
Moda (aprox): 50 tokens
------------------------------------------------



Map:   0%|          | 0/1006 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/aranes-Instructivo.pkl Instructivo: 1006


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1006
 }),
 <lowresource_llm_evaluation.LanguageDatasets.LanguageDataset at 0x7fec1784bc50>)

In [ ]:
import os
import pickle
import numpy as np
import zipfile
import io
from pathlib import Path
from collections import deque

# =====================================================================
# CONFIGURACIÓN
# =====================================================================
language = "gallego"
modelo_name = MODELO
tokenizer = TOKENIZER
thr = 0.3
compute_length = True
concatenate = False
max_tokens = 1024
N_max_lengths = 20000
max_length = 160

# Archivos específicos a utilizar
TARGET_FILES = {
    "press_and_blogs.txt", 
    "Wikipedia_gl.txt", 
    "Xunta_de_Galicia_contidos_web.txt", 
    "Coleccion_clasicos_servizo_publicacions_USC.txt", 
    "Nos_Diario.txt"
}

url_zenodo_zip = "https://zenodo.org/records/10687642/files/corpusnos.zip?download=1" 
zip_local_path = "corpus_nos.zip"
target_chunks = 400000 # Ahora el objetivo son 400k párrafos/fragmentos

# =====================================================================
# PROCESAMIENTO: FILTRADO Y CREACIÓN DE PÁRRAFOS
# =====================================================================

estructura = {}

if not os.path.exists(zip_local_path):
    print("Descargando ZIP...")
    import urllib.request
    urllib.request.urlretrieve(url_zenodo_zip, zip_local_path)

print("Procesando archivos seleccionados...")
with zipfile.ZipFile(zip_local_path, 'r') as z:
    for file_path in z.namelist():
        file_name = Path(file_path).name
        
        # Solo procesamos los archivos que nos interesan
        if file_name in TARGET_FILES:
            tema = file_name # Usamos el nombre del archivo como identificador único
            if tema not in estructura:
                estructura[tema] = deque()
            
            with z.open(file_path) as f:
                with io.TextIOWrapper(f, encoding='utf-8') as text_file:
                    buffer = []
                    current_len = 0
                    
                    for linea in text_file:
                        l_clean = linea.strip()
                        if not l_clean: continue
                        
                        buffer.append(l_clean)
                        current_len += len(l_clean) + 1 # +1 por el \n
                        
                        # Cuando alcanzamos el tamaño objetivo (aprox 160-200 chars)
                        if current_len >= 160:
                            estructura[tema].append("\n".join(buffer))
                            buffer = []
                            current_len = 0
                            
                        # Si llegamos a un máximo para no crear bloques gigantes
                        if current_len >= 300:
                            estructura[tema].append("\n".join(buffer))
                            buffer = []
                            current_len = 0

            # Mezclamos un poco los bloques creados de este archivo
            tmp_list = list(estructura[tema])
            np.random.shuffle(tmp_list)
            estructura[tema] = deque(tmp_list)

# =====================================================================
# SELECCIÓN ROUND-ROBIN (Equilibrio entre fuentes)
# =====================================================================
frases_finales = []
fuentes_disponibles = list(estructura.keys())

print("Seleccionando datos equilibrados...")
while len(frases_finales) < target_chunks and fuentes_disponibles:
    for fuente in list(fuentes_disponibles):
        if estructura[fuente]:
            frases_finales.append(estructura[fuente].popleft())
            if len(frases_finales) >= target_chunks:
                break
        else:
            fuentes_disponibles.remove(fuente)

del estructura # Liberamos RAM
print(f"Total de fragmentos coherentes creados: {len(frases_finales)}")

# =====================================================================
# PIPELINE CON ENMASCARADO
# =====================================================================
MASK_TOKEN = " <N_MASK> " # Un token único que no aparece en el texto normal

root_dir = "TrainDatasets"
save_dir = f"{root_dir}/{modelo_name.split('/')[-1]}"
os.makedirs(save_dir, exist_ok=True)

# 1. ENMASCARAR: Sustituimos \n por un token especial antes de cargar
frases_masked = [f.replace('\n', MASK_TOKEN) for f in frases_finales]

ds = LanguageDataset(language)
ds.read_list(frases_masked, "corpus_nos_filtrado")

# 2. FILTRAR: Ahora FastText no encontrará \n y no dará error
print("Filtrando por idioma...")
ds.filter_by_language(top_k=10, filter_language_thr=thr, batch_size=1024)

# 3. DESENMASCARAR: Restauramos el salto de línea original
ds.json = [{"text": f["text"].replace(MASK_TOKEN, '\n')} for f in ds.json]
print("¡Texto desenmascarado correctamente!")

if compute_length:
    lengths = ds.get_stats(tokenizer, N_max=N_max_lengths)

if max_length is None and lengths is not None:
    max_length = int(np.percentile(lengths, 95))
    print(f"🎯 max_length P95 ajustado: {max_length}")

train = ds.tokenize(tokenizer=tokenizer, max_length=max_length)
ds_gallego = ds

save_path = f"{save_dir}/{language}.pkl"
with open(save_path, "wb") as f:
    pickle.dump(train, f, protocol=5)

if os.path.exists(zip_local_path):
    os.remove(zip_local_path)

print(f"\nDataset guardado en: {save_path}")

Descargando ZIP...
Procesando archivos seleccionados...


In [5]:
saveInstructivo(
    "gallego",
    MODELO, # Nombre para la carpeta
    TOKENIZER,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=160
)

Leyendo archivo Instructivo/gallego.csv...
Leyendo archivo Instructivo_QA/gallego.csv...

📊 Analizando estadísticas de tokens (N=999)...
------------------------------------------------
Total líneas en dataset: 999
Media: 311.26 | Mediana: 208.00
Percentil 95: 982.70 (recomendado para max_length)
Percentil 98: 1151.80
Máximo: 1621 | Mínimo: 19
Moda (aprox): 30 tokens
------------------------------------------------



Map:   0%|          | 0/999 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/gallego-Instructivo.pkl Instructivo: 999


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 999
 }),
 <lowresource_llm_evaluation.LanguageDatasets.LanguageDataset at 0x7fec154db550>)

In [ ]:
import random
import time
import pandas as pd
from groq import Groq
from lowresource_llm_evaluation.generateDataset import generateInstructivoQADataset, generateInstructivoDataset
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset

def quitarYaProcesados(dataset: LanguageDataset, anotadas: pd.DataFrame):
    """
    Filtra el dataset original eliminando los textos que ya tienen
    un instructivo/QA generado en el CSV.
    
    El CSV tiene columna "text" con formato:
    
    <|user|>
    INSTRUCCIÓN + TEXTO ORIGINAL
    <|assistant|>
    RESPUESTA
    """

    ya = set()

    # Extraer el texto original de cada ejemplo generado
    for full in anotadas["text"].astype(str):
        try:
            # El texto original está en la parte del <|user|>
            user_block = full.split("<|user|>")[1].split("<|assistant|>")[0].strip()

            # El original es SIEMPRE la última línea del bloque del usuario
            original = user_block.split("\n")[-1].strip()

            if original:
                ya.add(original)
        except:
            continue

    # Filtrar dataset original
    res = LanguageDataset(dataset.language)
    res.json = [entry for entry in dataset.json if entry["text"].strip() not in ya]

    return res


ast, oc, gl = (LanguageDataset(l, initializeTatoeba=True) for l in ["asturiano","aranes","gallego"])
for lang, dataset in [("asturiano", ast), ("aranes", oc), ("gallego", gl)]:
    print(f"\n=== Empezando {lang} ===")


    os.makedirs("TrainDatasets/Instructivo", exist_ok=True)
    os.makedirs("TrainDatasets/Instructivo_QA", exist_ok=True)
    out_instruct = f"TrainDatasets/Instructivo/{lang}.csv"
    out_qa       = f"TrainDatasets/Instructivo_QA/{lang}.csv"


    # Crear CSVs vacíos si no existen
    if not os.path.exists(out_instruct):
        print("Nuevo")
        pd.DataFrame(columns=["text"]).to_csv(out_instruct, index=False)
    if not os.path.exists(out_qa):
        pd.DataFrame(columns=["text"]).to_csv(out_qa, index=False)

    dataset.json = dataset[:500]
    
    # Cargar existentes
    df_inst_exist = pd.read_csv(out_instruct)
    df_qa_exist   = pd.read_csv(out_qa)
    print(df_inst_exist.shape, df_qa_exist.shape)
    
    # Filtrar dataset para no repetir
    dataset_inst = quitarYaProcesados(dataset, df_inst_exist)
    dataset_qa   = dataset
    dataset_qa.json = dataset_qa[df_qa_exist.shape[0]:]

    # Generar instructivos
    if df_inst_exist.shape[0] < 500:
        print(f"Generando instructivos para {lang}...")
        nuevos_inst = generateInstructivoDataset(
            dataset_inst,
            api_key=os.getenv("GROQ_API_KEY"),
            model="openai/gpt-oss-120b",
            save=False,
            N= 500 - df_inst_exist.shape[0]
        )

    if df_qa_exist.shape[0] < 500:
        # Generar QA
        print(f"Generando QA para {lang}...")
        nuevos_qa = generateInstructivoQADataset(
            dataset_qa,
            api_key=os.getenv("GROQ_API_KEY"),
            model="openai/gpt-oss-120b",
            save=False,
            N=500 - df_qa_exist.shape[0]
        )

    # Guardar concatenado
    if df_inst_exist.shape[0] < 500:
        df_final_inst = pd.concat([df_inst_exist, pd.DataFrame({"text": nuevos_inst})], ignore_index=True)
        df_final_inst.to_csv(out_instruct, index=False)
        print(f"✓ Nuevos instructivos: {len(nuevos_inst)}")
    if df_qa_exist.shape[0] < 500:
        df_final_qa = pd.concat([df_qa_exist, pd.DataFrame({"text": nuevos_qa})], ignore_index=True)
        df_final_qa.to_csv(out_qa, index=False)

        print(f"✓ Nuevos QA: {len(nuevos_qa)}")


print("\n=== Generación completada ===")


Descargando tatoeba para asturiano:
Completado con éxito
Descargando tatoeba para aranes:


Completado con éxito
Descargando tatoeba para gallego:


Completado con éxito

=== Empezando asturiano ===
(724, 1) (458, 1)
Generando QA para asturiano...

Dataset QA generado (asturiano): 0 ejemplos válidos.
✓ Nuevos QA: 0

=== Empezando aranes ===


(506, 1) (500, 1)

=== Empezando gallego ===
(499, 1) (148, 1)
Generando instructivos para gallego...


Progreso: 100.0% (1/1) | step: 5 s, remaining time: 0 min 0 s
Dataset instructivo generado (gallego): 1 ejemplos válidos.
Generando QA para gallego...
Progreso:  10.8% (38/352) | step: 4 s, remaining time: 20 min 56 s
Error en intento 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7182, Requested 1199. Please try again in 2.8575s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Reintentando en 0.5 segundos...
Progreso:  21.6% (76/352) | step: 1 s, remaining time: 4 min 36 s ss
Error en intento 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7195, Requested 1406. Plea

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------------------------------------------
# 1. Cargar longitudes usando tu método get_stats()
# -------------------------------------------------------------------
# Suponiendo que ya tienes los LanguageDataset cargados:
ds = lambda x: (
        LanguageDataset(x)
        .read_opus(source="NLLB", version=1)
        .filter_by_language(top_k=10, filter_language_thr=0.3, batch_size=1024)
    )
ds_ast, ds_arn, ds_gl = ds("asturiano"), ds("aranes"), 
lengths_ast = ds_ast.get_stats(TOKENIZER, do_print=False)
lengths_arn = ds_arn.get_stats(TOKENIZER, do_print=False)
lengths_gl  = ds_gl.get_stats(TOKENIZER, do_print=False)

# Para este ejemplo, asumimos que ya existen:
# lengths_ast, lengths_arn, lengths_gl

# -------------------------------------------------------------------
# 2. Histograma solapado (transparente)
# -------------------------------------------------------------------
plt.figure(figsize=(10,6))
sns.histplot(lengths_ast, bins=80, color="blue", alpha=0.35, kde=False, label="Asturiano")
sns.histplot(lengths_arn, bins=80, color="red", alpha=0.35, kde=False, label="Aranés")
sns.histplot(lengths_gl,  bins=80, color="green", alpha=0.35, kde=False, label="Gallego")

plt.title("Distribución de longitudes (tokens) — Histogramas solapados")
plt.xlabel("Longitud en tokens")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 3. KDE comparado (curvas suaves)
# -------------------------------------------------------------------
plt.figure(figsize=(10,6))
sns.kdeplot(lengths_ast, color="blue", label="Asturiano", linewidth=2)
sns.kdeplot(lengths_arn, color="red", label="Aranés", linewidth=2)
sns.kdeplot(lengths_gl,  color="green", label="Gallego", linewidth=2)

plt.title("Distribución de longitudes — KDE comparado")
plt.xlabel("Longitud en tokens")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 4. Violinplot comparado (muy visual)
# -------------------------------------------------------------------
plt.figure(figsize=(10,6))
sns.violinplot(
    data=[lengths_ast, lengths_arn, lengths_gl],
    palette=["blue", "red", "green"],
    cut=0
)
plt.xticks([0,1,2], ["Asturiano", "Aranés", "Gallego"])
plt.title("Distribución de longitudes — Violinplot")
plt.ylabel("Tokens")
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 5. Gráfica de percentiles (P50, P75, P90, P95, P98)
# -------------------------------------------------------------------
percentiles = [50, 75, 90, 95, 98]

stats_ast = np.percentile(lengths_ast, percentiles)
stats_arn = np.percentile(lengths_arn, percentiles)
stats_gl  = np.percentile(lengths_gl,  percentiles)

df_percentiles = pd.DataFrame({
    "Percentil": percentiles,
    "Asturiano": stats_ast,
    "Aranés": stats_arn,
    "Gallego": stats_gl
})

plt.figure(figsize=(10,6))
sns.lineplot(data=df_percentiles, x="Percentil", y="Asturiano", marker="o", label="Asturiano")
sns.lineplot(data=df_percentiles, x="Percentil", y="Aranés", marker="o", label="Aranés")
sns.lineplot(data=df_percentiles, x="Percentil", y="Gallego", marker="o", label="Gallego")

plt.title("Comparación de percentiles por lengua")
plt.ylabel("Tokens")
plt.xlabel("Percentil")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
